<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/text_match.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

TIME_PATTERN = re.compile(r'(\d{1,2})\s*[:시]\s*(\d{1,2})?')


def extract_report_hour(message: str):
    """SMS 원문에서 '오늘 12:45' '14:12분경' '09시 46분경' 같은 시각 표현의 시(hour)를 뽑는다."""
    if not isinstance(message, str):
        return None
    m = TIME_PATTERN.search(message)
    if not m:
        return None
    hour = int(m.group(1))
    return hour if 0 <= hour <= 23 else None


def match_sms_to_call119(sms_row: pd.Series, call119_df: pd.DataFrame) -> dict:

    ymd = str(sms_row["date"]).replace("-", "")
    first_region = str(sms_row["region"]).split(",")[0]
    tokens = first_region.split()

    sido = tokens[0] if len(tokens) > 0 else None
    sigungu = tokens[1] if len(tokens) > 1 else None
    emd = tokens[2] if len(tokens) > 2 else None

    if not sigungu or sigungu == "전체":
        return None  # 광역(시 전체) 알림은 별도 처리 필요 - 여기선 매칭 대상 아님

    base = call119_df[
        (call119_df["dclr_ymd"].astype(str) == ymd)
        & (call119_df["sido"] == sido)
        & (call119_df["sigungu"] == sigungu)
    ]

    if len(base) == 0:
        return None  # 정보부족(위험 아님) - final_alert_score에서 중립 처리 대상

    match_level = "sigungu"
    candidates = base

    # 읍면동까지 일치하면 우선 적용
    if emd:
        emd_cand = base[base["emd"] == emd]
        if len(emd_cand) > 0:
            candidates = emd_cand
            match_level = "emd"

    # 재난유형(화재 등) 일치로 추가 좁히기
    disaster_type = sms_row.get("disaster_type")
    if isinstance(disaster_type, str):
        type_cand = candidates[candidates["incident_type"].str.contains(disaster_type, na=False)]
        if len(type_cand) > 0:
            candidates = type_cand

    # 시각 근접도로 최종 순위 (텍스트 유사도 대체 신호)
    sms_hour = extract_report_hour(sms_row.get("message"))
    best_report_id = None
    hour_diff = None

    if sms_hour is not None and "dclr_hr" in candidates.columns:
        diffs = (candidates["dclr_hr"] - sms_hour).abs()
        diffs = diffs.apply(lambda d: min(d, 24 - d))  # 자정 넘어가는 경우 보정
        best_idx = diffs.idxmin()
        best_report_id = candidates.loc[best_idx, "report_id"]
        hour_diff = float(diffs.loc[best_idx])
    else:
        best_report_id = candidates.iloc[0]["report_id"]

    return {
        "match_level": match_level,
        "candidate_count": len(candidates),
        "best_report_id": best_report_id,
        "hour_diff": hour_diff,
    }


# ---------------------------------------------------------------------------
# 검증
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    sms = pd.read_csv("/content/drive/MyDrive/REDRED/disaster_sms_preprocessed.csv")
    call119 = pd.read_csv("/content/drive/MyDrive/REDRED/busan_119_2024_dates.csv")
    call119["dclr_ymd"] = call119["dclr_ymd"].astype(str)

    busan_fire = sms[(sms["region"].str.contains("부산", na=False)) & (sms["disaster_type"] == "화재")]

    matched = 0
    for _, row in busan_fire.iterrows():
        result = match_sms_to_call119(row, call119)
        status = "매칭없음(정보부족)" if result is None else \
            f"매칭됨 [{result['match_level']}, 후보 {result['candidate_count']}건, 시간차 {result['hour_diff']}시간]"
        if result is not None:
            matched += 1
        print(f"{row['date']} {row['region'].split(',')[0]:20s} {status} | {row['message'][:35]}")

    print(f"\n=== 매칭: {matched} / {len(busan_fire)} ===")


Mounted at /content/drive
2024-01-21 부산광역시 금정구            매칭됨 [sigungu, 후보 2건, 시간차 2.0시간] | 1.21.(일) 14:12분경 동래구 온천동 425-10번지 빌
2024-01-21 부산광역시 전체             매칭없음(정보부족) | 최근 주거시설 내 화재가 빈번하오니, 전열기, 가스밸브 안전을 
2024-02-05 부산광역시 사상구            매칭됨 [sigungu, 후보 3건, 시간차 1.0시간] | 오늘 10:53분 주례동 경남정보대학교에서 화재 발생하여 진화중
2024-08-01 부산광역시 금정구            매칭됨 [sigungu, 후보 24건, 시간차 0.0시간] | 06:40 욱성화학 화재 관련하여 화재현장에 폭발위험은 전혀 없
2024-08-01 부산광역시 금정구            매칭됨 [sigungu, 후보 24건, 시간차 0.0시간] | 06:40 시 회동동 유성화학에서 화재 발생 만약 피부 접촉 시
2024-08-01 부산광역시 금정구            매칭됨 [sigungu, 후보 24건, 시간차 None시간] | 오늘 부산시 회동동 욱성화확건물에서 화재 발생.연기가 많이 발생
2024-08-01 부산광역시 금정구            매칭됨 [sigungu, 후보 24건, 시간차 0.0시간] | 오늘 06:40경 부산시 회동동동 육성화확에 화재 사고 발생. 
2024-08-01 부산광역시 금정구            매칭됨 [sigungu, 후보 24건, 시간차 0.0시간] | 오늘 06:40경 금정구 회동동 욱성화학에서 화재가 발생하여 연
2024-10-24 부산광역시 남구             매칭됨 [sigungu, 후보 2건, 시간차 None시간] | 금일 동구 55보급창에서 화재가 발생하여 연기, 분진이 다량 발
2024-10-24 부산광역시 남구             매칭됨 [sigungu, 후보 2건, 시간차 None시간] | 금일 